# 🎯 CosyVoice 2 — Test Notebook
**BookVoice-AI Engine Test**

Teste CosyVoice 2 Qualität für türkisch/deutsch/englisch

In [ ]:
#@title ⚙️ Schritt 1: CosyVoice 2 installieren
#@markdown Dauert ~5 Minuten
import subprocess, sys, os

print('📦 System Pakete...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg', 'sox', 'libsox-dev'], capture_output=True)
print('✅ System OK!')

print('📦 CosyVoice 2 installieren...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cosyvoice2-eu', 'fastapi', 'uvicorn', 'python-multipart'], capture_output=True)
print('✅ Installation OK!')

print('📦 Modell laden (~3GB)...')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
from cosyvoice2eu import CosyVoice2
model = CosyVoice2('FunAudioLLM/CosyVoice2-0.5B')
print('✅ Modell bereit!')

print('\n🎉 INSTALLATION FERTIG!')


In [ ]:
#@title 🎤 Schritt 2: Stimme hochladen
#@markdown WAV/MP3 Datei hochladen (10-30 Sekunden)
from google.colab import files
import os

os.makedirs('/content/stimmen', exist_ok=True)
print('📂 Stimme hochladen:')
uploaded = files.upload()
voice_file = None
for filename, data in uploaded.items():
    path = f'/content/stimmen/{filename}'
    with open(path, 'wb') as f:
        f.write(data)
    voice_file = path
    print(f'✅ Stimme gespeichert: {filename}')


In [ ]:
#@title 🎙️ Schritt 3: Test generieren
test_text = 'Bu bir test metnidir. CosyVoice 2 ile Türkçe ses kalitesini test ediyoruz.' #@param {type:"string"}
sprache = 'tr' #@param ['tr', 'de', 'en']

import torchaudio, torch
from IPython.display import Audio, display

print(f'🎙️ Generiere: {test_text[:50]}...')

# Stimme laden
prompt_speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    prompt_speech = torchaudio.functional.resample(prompt_speech, sr, 16000)

# Generieren
output = model.inference_zero_shot(
    test_text,
    'Referenz Stimme für Voice Cloning',
    prompt_speech
)

# Speichern
out_path = '/content/test_cosyvoice.wav'
torchaudio.save(out_path, output['tts_speech'], 22050)
print('✅ Fertig! Anhören:')
display(Audio(out_path))


In [ ]:
#@title 📊 Schritt 4: Qualitätstest verschiedene Texte
from IPython.display import Audio, display
import torchaudio

test_texte = {
    'Türkisch': 'Bismillahirrahmanirrahim. Rahman ve Rahim olan Allah\'ın adıyla.',
    'Deutsch': 'Guten Tag! Dies ist ein Test der CosyVoice 2 Sprachsynthese auf Deutsch.',
    'Englisch': 'Hello! This is a test of CosyVoice 2 text to speech synthesis in English.'
}

prompt_speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    prompt_speech = torchaudio.functional.resample(prompt_speech, sr, 16000)

for lang, text in test_texte.items():
    print(f'\n🎙️ {lang}:')
    output = model.inference_zero_shot(text, 'Referenz', prompt_speech)
    path = f'/content/test_{lang.lower()}.wav'
    torchaudio.save(path, output['tts_speech'], 22050)
    print(f'✅ {lang} fertig:')
    display(Audio(path))
